# 01 — Dataset Preparation & EDA
**Drone Detection | ITMO Diploma Thesis**

Этот ноутбук:
1. Монтирует Google Drive
2. Скачивает датасет через Roboflow API
3. Проводит EDA (анализ данных)
4. Создаёт `data.yaml` для YOLO
5. Конвертирует в COCO формат для Faster R-CNN

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
!uv pip install ultralytics roboflow pycocotools opencv-python-headless tqdm matplotlib seaborn
import ultralytics; ultralytics.checks()

In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── CELL 3: Imports & path variables ─────────────────────────────────────────
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# ── Path configuration ────────────────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/Colab Notebooks')
DATA_DIR    = DRIVE_ROOT / 'prepared' / 'yolo'       # YOLO (папка prepared на Drive)
COCO_DIR    = DRIVE_ROOT / 'prepared' / 'dataset_coco'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
RESULTS_DIR = DRIVE_ROOT / 'results'
VIDEO_DIR   = DRIVE_ROOT / 'videos'

for d in [DATA_DIR, COCO_DIR, WEIGHTS_DIR, RESULTS_DIR, VIDEO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
CLASS_COLORS = {'DRONE': '#e74c3c', 'AIRPLANE': '#3498db', 'HELICOPTER': '#2ecc71', 'BIRD': '#f1c40f'}
print('Paths ready.')

## 1. Скачать датасет с Roboflow

> **Инструкция**: Зайди на https://universe.roboflow.com, найди проект с нужными классами,
> нажми Export → YOLOv8 format → Show download code → скопируй API ключ и строки ниже.

Рекомендуемые датасеты для объединения:
- `drone-detection-6dp1d` (4k+ imgs)
- `uav-bird-detection` (3k+ imgs)
- `aerial-object-detection` (2k+ imgs)

In [ ]:
# ── CELL 4: Download from Roboflow ────────────────────────────────────────────
# Замени RF_API_KEY на свой ключ из https://app.roboflow.com/settings/api
RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # <-- ВСТАВЬ СЮДА

from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)

# Скачать объединённый датасет (после merge на Roboflow Universe)
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# dataset = project.version(1).download("yolov8", location=str(DATA_DIR))

# ── Альтернатива: скачать отдельные датасеты и объединить ────────────────────
# dataset1 = rf.workspace("ws1").project("drone-detection").version(1).download("yolov8")
# dataset2 = rf.workspace("ws2").project("uav-bird").version(1).download("yolov8")

print('Dataset download complete.')
print(f'Location: {DATA_DIR}')

In [ ]:
# ── CELL 5: Merge multiple datasets ──────────────────────────────────────────
def merge_yolo_datasets(source_dirs: list[str], target_dir: Path) -> None:
    """Merge multiple YOLO format datasets into one target directory.

    Args:
        source_dirs: List of paths to source dataset root dirs.
        target_dir: Target directory for merged dataset.
    """
    for split in ['train', 'val', 'test']:
        (target_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (target_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for src_idx, src_dir in enumerate(source_dirs):
        src = Path(src_dir)
        for split in ['train', 'valid', 'val', 'test']:
            split_name = 'val' if split == 'valid' else split
            img_dir = src / 'images' / split
            lbl_dir = src / 'labels' / split
            if not img_dir.exists():
                continue
            for img_path in tqdm(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')),
                                 desc=f'src{src_idx}/{split_name}'):
                stem = f's{src_idx}_{img_path.stem}'
                shutil.copy(img_path, target_dir / 'images' / split_name / f'{stem}{img_path.suffix}')
                lbl_path = lbl_dir / f'{img_path.stem}.txt'
                if lbl_path.exists():
                    shutil.copy(lbl_path, target_dir / 'labels' / split_name / f'{stem}.txt')

    print(f'Merge complete → {target_dir}')


# Раскомментируй и вставь пути к скачанным датасетам:
# merge_yolo_datasets(
#     source_dirs=['/content/dataset1', '/content/dataset2'],
#     target_dir=DATA_DIR
# )

In [ ]:
# ── CELL 6: Remap class IDs to standard order ─────────────────────────────────
# Используй это если в исходных датасетах разные индексы классов
# TARGET: 0=DRONE, 1=AIRPLANE, 2=HELICOPTER, 3=BIRD

SOURCE_TO_TARGET: dict[int, int] = {
    # Пример: если в источнике BIRD=0, DRONE=1, AIRPLANE=2, HELICOPTER=3
    # 0: 3,  # BIRD
    # 1: 0,  # DRONE
    # 2: 1,  # AIRPLANE
    # 3: 2,  # HELICOPTER
}

def remap_labels(labels_dir: Path, mapping: dict[int, int]) -> None:
    """Remap class IDs in all YOLO label .txt files.

    Args:
        labels_dir: Root of labels/ directory (contains train/val/test).
        mapping: Dict of {old_class_id: new_class_id}.
    """
    if not mapping:
        print('No remapping needed.')
        return
    for split in ['train', 'val', 'test']:
        split_dir = labels_dir / split
        if not split_dir.exists():
            continue
        for lbl_file in tqdm(list(split_dir.glob('*.txt')), desc=f'Remap {split}'):
            lines = lbl_file.read_text().strip().splitlines()
            new_lines = []
            for line in lines:
                parts = line.split()
                if not parts:
                    continue
                old_cls = int(parts[0])
                new_cls = mapping.get(old_cls, old_cls)
                new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
            lbl_file.write_text('\n'.join(new_lines))
    print('Remap done.')


# remap_labels(DATA_DIR / 'labels', SOURCE_TO_TARGET)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── CELL 7: Count images and labels per split ────────────────────────────────
def dataset_stats(data_dir: Path) -> dict:
    """Compute per-split image counts and class distribution.

    Args:
        data_dir: Root of YOLO format dataset.

    Returns:
        Dict with stats per split.
    """
    stats = {}
    for split in ['train', 'val', 'test']:
        img_dir = data_dir / 'images' / split
        lbl_dir = data_dir / 'labels' / split
        if not img_dir.exists():
            continue
        images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        class_counts = Counter()
        for lbl in tqdm(list(lbl_dir.glob('*.txt')), desc=split, leave=False):
            for line in lbl.read_text().strip().splitlines():
                if line.strip():
                    class_counts[int(line.split()[0])] += 1
        stats[split] = {'images': len(images), 'objects': dict(class_counts)}
    return stats


stats = dataset_stats(DATA_DIR)
for split, s in stats.items():
    print(f'\n{split.upper()}: {s["images"]} images')
    total_obj = sum(s['objects'].values())
    for cls_id, cnt in sorted(s['objects'].items()):
        name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'class_{cls_id}'
        print(f'  {name}: {cnt} ({cnt/total_obj*100:.1f}%)')

In [ ]:
# ── CELL 8: Plot class distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')

for ax, split in zip(axes, ['train', 'val', 'test']):
    if split not in stats:
        ax.set_visible(False)
        continue
    obj = stats[split]['objects']
    labels = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else f'cls_{i}' for i in sorted(obj)]
    values = [obj[i] for i in sorted(obj)]
    colors = [CLASS_COLORS.get(l, '#95a5a6') for l in labels]
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{split.capitalize()} ({stats[split]["images"]} images)')
    ax.set_ylabel('Object count')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
save_path = RESULTS_DIR / 'class_distribution.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved → {save_path}')
plt.show()

In [ ]:
# ── CELL 9: Image size distribution ──────────────────────────────────────────
def get_image_sizes(img_dir: Path, n_samples: int = 500) -> list[tuple[int, int]]:
    """Sample image sizes from a directory.

    Args:
        img_dir: Directory containing images.
        n_samples: Max number of images to sample.

    Returns:
        List of (width, height) tuples.
    """
    all_imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    sample = random.sample(all_imgs, min(n_samples, len(all_imgs)))
    sizes = []
    for p in tqdm(sample, desc='Reading sizes'):
        try:
            with Image.open(p) as im:
                sizes.append(im.size)  # (width, height)
        except Exception:
            pass
    return sizes


train_sizes = get_image_sizes(DATA_DIR / 'images' / 'train')
widths  = [s[0] for s in train_sizes]
heights = [s[1] for s in train_sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='#3498db', edgecolor='white')
axes[0].set_title('Image Width Distribution (train)')
axes[0].set_xlabel('Width (px)')
axes[1].hist(heights, bins=30, color='#2ecc71', edgecolor='white')
axes[1].set_title('Image Height Distribution (train)')
axes[1].set_xlabel('Height (px)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'image_sizes.png', dpi=150)
plt.show()
print(f'Width:  mean={np.mean(widths):.0f}  min={min(widths)}  max={max(widths)}')
print(f'Height: mean={np.mean(heights):.0f}  min={min(heights)}  max={max(heights)}')

In [ ]:
# ── CELL 10: BBox size distribution ──────────────────────────────────────────
def get_bbox_sizes(labels_dir: Path, split: str = 'train',
                   n_samples: int = 2000) -> dict[str, list[float]]:
    """Collect normalized bbox widths and heights per class.

    Args:
        labels_dir: Root labels/ directory.
        split: Dataset split to analyse.
        n_samples: Max label files to read.

    Returns:
        Dict mapping class_name → list of (w*h) relative areas.
    """
    label_files = list((labels_dir / split).glob('*.txt'))
    sample = random.sample(label_files, min(n_samples, len(label_files)))
    class_areas: dict[str, list[float]] = defaultdict(list)
    for lf in sample:
        for line in lf.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            w, h = float(parts[3]), float(parts[4])
            name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls_{cls_id}'
            class_areas[name].append(w * h)
    return dict(class_areas)


bbox_data = get_bbox_sizes(DATA_DIR / 'labels')

fig, ax = plt.subplots(figsize=(10, 5))
for cls_name, areas in bbox_data.items():
    ax.hist(areas, bins=50, alpha=0.6, label=cls_name, color=CLASS_COLORS.get(cls_name, '#95a5a6'))
ax.set_xlabel('Relative BBox area (w×h, normalized)')
ax.set_ylabel('Count')
ax.set_title('BBox Area Distribution by Class (train)')
ax.legend()
ax.set_xlim(0, 0.15)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bbox_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 11: Visualise random annotated samples ───────────────────────────────
def visualize_samples(data_dir: Path, split: str = 'train', n: int = 9) -> None:
    """Display a grid of random annotated images.

    Args:
        data_dir: Dataset root.
        split: Split to sample from.
        n: Number of images to display.
    """
    img_dir = data_dir / 'images' / split
    lbl_dir = data_dir / 'labels' / split
    all_imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    sample = random.sample(all_imgs, min(n, len(all_imgs)))

    cols = 3
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for ax, img_path in zip(axes, sample):
        img = np.array(Image.open(img_path).convert('RGB'))
        h, w = img.shape[:2]
        ax.imshow(img)
        lbl_path = lbl_dir / f'{img_path.stem}.txt'
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls_{cls_id}'
                color = CLASS_COLORS.get(name, '#ffffff')
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                         linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 3, name, fontsize=8, color=color,
                        bbox=dict(facecolor='black', alpha=0.5, pad=1))
        ax.axis('off')
        ax.set_title(img_path.name[:30], fontsize=7)

    for ax in axes[len(sample):]:
        ax.set_visible(False)

    plt.suptitle(f'Sample annotated images ({split})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'samples_{split}.png', dpi=120)
    plt.show()


visualize_samples(DATA_DIR, split='train', n=9)

## 3. Создать data.yaml и конвертировать в COCO

In [ ]:
# ── CELL 12: Write data.yaml ──────────────────────────────────────────────────
import yaml

data_yaml = {
    'path': str(DATA_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

yaml_path = DATA_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f'data.yaml → {yaml_path}')
print(yaml.dump(data_yaml, default_flow_style=False))

In [ ]:
# ── CELL 13: Convert YOLO → COCO JSON ────────────────────────────────────────
def yolo_to_coco(yolo_data_dir: Path, coco_output_dir: Path,
                 class_names: list[str]) -> None:
    """Convert YOLO format dataset to COCO JSON format.

    Args:
        yolo_data_dir: Root of YOLO dataset (contains images/ and labels/).
        coco_output_dir: Output directory for COCO format.
        class_names: List of class names in order.
    """
    categories = [{'id': i, 'name': n, 'supercategory': 'aerial'}
                  for i, n in enumerate(class_names)]

    for split in ['train', 'val', 'test']:
        img_src = yolo_data_dir / 'images' / split
        lbl_src = yolo_data_dir / 'labels' / split
        if not img_src.exists():
            print(f'Skip {split} — not found')
            continue

        img_dst = coco_output_dir / 'images' / split
        ann_dst = coco_output_dir / 'annotations'
        img_dst.mkdir(parents=True, exist_ok=True)
        ann_dst.mkdir(parents=True, exist_ok=True)

        coco: dict = {'images': [], 'annotations': [], 'categories': categories}
        ann_id = 1

        all_imgs = sorted(list(img_src.glob('*.jpg')) + list(img_src.glob('*.png')))
        for img_id, img_path in enumerate(tqdm(all_imgs, desc=f'COCO {split}'), start=1):
            with Image.open(img_path) as im:
                w, h = im.size

            shutil.copy(img_path, img_dst / img_path.name)

            coco['images'].append({
                'id': img_id, 'file_name': img_path.name,
                'width': w, 'height': h,
            })

            lbl_path = lbl_src / f'{img_path.stem}.txt'
            if not lbl_path.exists():
                continue

            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x_min = (cx - bw / 2) * w
                y_min = (cy - bh / 2) * h
                pw, ph = bw * w, bh * h
                area = pw * ph
                coco['annotations'].append({
                    'id': ann_id, 'image_id': img_id, 'category_id': cls_id,
                    'bbox': [round(x_min, 2), round(y_min, 2), round(pw, 2), round(ph, 2)],
                    'area': round(area, 2), 'iscrowd': 0,
                })
                ann_id += 1

        out_json = ann_dst / f'instances_{split}.json'
        with open(out_json, 'w') as f:
            json.dump(coco, f)
        print(f'{split}: {len(coco["images"])} imgs, {len(coco["annotations"])} anns → {out_json}')


yolo_to_coco(DATA_DIR, COCO_DIR, CLASS_NAMES)
print('\nCOCO conversion done!')

In [ ]:
# ── CELL 14: Summary ──────────────────────────────────────────────────────────
print('=' * 60)
print('DATASET PREPARATION COMPLETE')
print('=' * 60)
total = sum(s['images'] for s in stats.values())
print(f'Total images: {total}')
for split, s in stats.items():
    print(f'  {split}: {s["images"]} images')
print(f'\nYOLO dataset: {DATA_DIR}')
print(f'COCO dataset: {COCO_DIR}')
print(f'data.yaml:    {DATA_DIR / "data.yaml"}')
print(f'\nNext step → 02_yolov12_train.ipynb')